In [9]:
from src.config import cfg
import re
import os
import string
import unicodedata
from collections import Counter
from datasets import load_dataset
from itertools import chain

In [2]:
ds = load_dataset("parquet", data_files=cfg.dataset.path, split="train")

In [3]:
def clean_text(text):
    text = unicodedata.normalize("NFC", text)
    text = text.lower().replace("i\u0307", "i")

    aylar = "ocak|şubat|mart|nisan|mayıs|haziran|temmuz|ağustos|eylül|ekim|kasım|aralık"

    # "18 ağustos 1227" gibi tarihleri tek tokena çevirir
    text = re.sub(rf"\b\d{{1,2}}\s+({aylar})\s+\d{{3,4}}\b", " <|tarih|> ", text)

    # "1206-1227", "1914 – 1918" gibi sayı aralıklarını tek tokena çevirir
    text = re.sub(r"\b\d+\s*[-–—]\s*\d+\b", " <|sayi_araligi|> ", text)

    # "2023te", "000inden", "5inci" gibi sayı+ek yapılarını sayı tokenı ve ek olarak ayırır
    text = re.sub(r"\b(\d+)([a-zçğıöşü]+)\b", r" <|sayi|> \2 ", text)

    # kalan tekil sayıları sayı tokenına çevirir
    text = re.sub(r"\b\d+\b", " <|sayi|> ", text)

    # apostrof ve tırnak işaretlerini siler, kelimeleri bölmez
    text = re.sub(r"[\'\"`’‘“”]", "", text)

    # kelime arasındaki tire dahil tüm tireleri boşluğa çevirir
    text = re.sub(r"[-–—]", " ", text)

    # harf, sayı, alt çizgi, boşluk ve özel token karakterleri dışındaki noktalama işaretlerini temizler
    text = re.sub(r"[^\w\s<>|çğıöşü]", " ", text)

    # fazla boşlukları teke indirir
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [4]:
def test_clean_text():
    examples = [
        "Cengiz Han (doğum adıyla Temuçin, 1162 – 18 Ağustos 1227), Moğol İmparatorluğu",
        "Cengiz Han 1206-1227 yılları arasında hüküm sürdü.",
        "kelime'lerin arasındaki “tırnakları” sil.",
        "Fatih Sultan Mehmed 30 Mart 1432 tarihinde doğdu.",
        "000e",
        "000er",
        "000i",
        "000in",
        "000inci",
        "000inden",
        "000ine",
        "5inci yüzyıl",
        "2023te seçim oldu",
        "12den büyük",
        "7ye bölünmez",
        "1914-1918 savaşı",
    ]

    for text in examples:
        print("INPUT :", text)
        print("OUTPUT:", clean_text(text))
        print("-" * 60)
test_clean_text()

INPUT : Cengiz Han (doğum adıyla Temuçin, 1162 – 18 Ağustos 1227), Moğol İmparatorluğu
OUTPUT: cengiz han doğum adıyla temuçin <|sayi|> <|tarih|> moğol imparatorluğu
------------------------------------------------------------
INPUT : Cengiz Han 1206-1227 yılları arasında hüküm sürdü.
OUTPUT: cengiz han <|sayi_araligi|> yılları arasında hüküm sürdü
------------------------------------------------------------
INPUT : kelime'lerin arasındaki “tırnakları” sil.
OUTPUT: kelimelerin arasındaki tırnakları sil
------------------------------------------------------------
INPUT : Fatih Sultan Mehmed 30 Mart 1432 tarihinde doğdu.
OUTPUT: fatih sultan mehmed <|tarih|> tarihinde doğdu
------------------------------------------------------------
INPUT : 000e
OUTPUT: <|sayi|> e
------------------------------------------------------------
INPUT : 000er
OUTPUT: <|sayi|> er
------------------------------------------------------------
INPUT : 000i
OUTPUT: <|sayi|> i
--------------------------------------

In [5]:
def process_data(ds=ds):
    path = cfg.dataset.path

    def process_batch(batch):
        cleaned = [clean_text(text) for text in batch["text"]]
        tokens = [text.split(" ") for text in cleaned]
        return {"text": cleaned, "tokens": tokens}

    num_proc = os.cpu_count()
    ds = ds.map(process_batch, batched=True, batch_size=1000, num_proc=num_proc)

    texts = ds["text"]
    tokens = ds["tokens"]

    return texts, tokens
data,tokens=process_data()

Map (num_proc=8):   0%|          | 0/1005338 [00:00<?, ? examples/s]

In [6]:
idx = 2
step = 0
ad=50
b,e = step*ad,step*ad+ad
print(tokens[idx][b:e])

['mehmed', 'mustafa', 'subhi', 'kısaca', 'mustafa', 'suphi', 'veya', 'bazı', 'kaynaklarda', 'kullanıldığı', 'haliyle', 'osmanlıca', 'yazıma', 'göre', 'mustafa', 'subhi', '<|tarih|>', 'veya', '<|tarih|>', '<|tarih|>', 'türk', 'komünist', 've', 'türkiye', 'komünist', 'partisinin', 'ilk', 'merkez', 'komitesi', 'başkanı', 'yaşamı', 'erken', 'dönemler', 'ailesi', 'aslen', 'samsunlu', 'bir', 'aileden', 'gelmektedir', '<|tarih|>', 'tarihli', 'sicill', 'i', 'ahvâl', 'defteri', 'no', '<|sayi|>', 'sayfa', '<|sayi|>', 'e']


In [10]:
counter = Counter(chain.from_iterable(tokens))

In [11]:
vocab = [w for w, count in counter.items() if count >= 5]
len(vocab)

499607

In [14]:
print(vocab[:50])

['cengiz', 'han', 'doğum', 'adıyla', 'temuçin', 'ya', 'da', 'timuçin', '<|sayi|>', 'ağustos', 'moğol', 'imparatorluğunun', 'kurucusu', 've', 'ilk', 'hanı', 'olan', 'komutan', 'hükümdardır', 'hükümdarlığı', 'döneminde', 'gerçekleştirdiği', 'hiçbir', 'savaşı', 'kaybetmeyen', 'dünya', 'tarihinin', 'en', 'büyük', 'askerî', 'liderlerinden', 'birisi', 'olarak', 'kabul', 'edilmektedir', 'bu', 'sebeple', 'ismi', 'çok', 'güçlü', 'hükümdar', 'anlamına', 'gelen', 'ismiyle', 'anılmaktadır', 'yüzyılın', 'başında', 'orta', 'asyadaki', 'tüm']
